# RAID-CF on Colab / Kaggle

Runs the unchanged `jobs/raid-cf/<dataset>.slurm` scripts with `bash`, setting by hand the three
SLURM variables they read. Only *where* the job runs changes - code, flags and workflow are the
same as on the cluster. The platform is detected automatically.

**Storage pattern (both platforms)**: the repo runs on local disk (stage 1 makes a symlink under
`checkpoint/`, and per-epoch `train_state.pth` writes are faster locally). `logs/` and `checkpoint/`
are rsynced to a backup dir every 15 min and once more when the job stops:

| | Colab | Kaggle |
|---|---|---|
| repo | `/content/MoDiCF` | `/tmp/MoDiCF` (kept out of the saved output) |
| data | `MyDrive/MoDiCF-data/<ds>/` | any attached Dataset holding `<ds>/train_mat` |
| backup | `MyDrive/MoDiCF-runs/` | `/kaggle/working/MoDiCF-runs/` (= the version's output) |
| restore from | `MyDrive/MoDiCF-runs/` | every `checkpoint/` + `logs/` found in attached inputs |
| job | background; monitor with cell 6 | blocking, stopped before the time limit |

**Resuming**
- Colab: rerun all cells with the same `DATASET / VARIANT / SEED`.
- Kaggle: *Save Version -> Save & Run All*. For the next chunk, attach this notebook's previous
  output as an input and Save & Run All again. Internet must be on (Settings) for `git clone` / `pip`.

Stage 2 resumes from `train_state.pth`; stage 1 (pretrain) has no resume, so provide the server's
pretrain dirs as real dirs (not symlinks): on Colab under `MyDrive/MoDiCF-runs/checkpoint/<ds>/`,
on Kaggle as a Dataset whose root holds `checkpoint/<ds>/...`.
`raid-cf` / `retrieval` need `gmddc-pretrain-k10-b0.3-seed<N>/`, `debias` needs `pretrain-seed<N>/`.

## 0. Config + platform detection

In [ ]:
import os, sys, time, glob, re, subprocess

DATASET = "baby"        # baby | tiktok | allrecipes
VARIANT = "raid-cf"     # retrieval | debias | raid-cf
SEED    = 0             # one seed per run (= SLURM_ARRAY_TASK_ID)
EXTRA_ARGS = ""         # optional positional args: "[lambda_cf] [group_k] [retrieval_k] [retrieval_beta]"

BRANCH = "develop/raid-cf"
REPO_URL = "https://github.com/thienenpi/MoDiCF.git"
MATCH_SERVER_TORCH = True   # install torch 2.3.1+cu118 like the server (dependencies.sh)
KAGGLE_BUDGET_H = 11.5      # Kaggle: stop the job this long after the notebook started (limit is 12h)
BACKUP_EVERY_S = 900

NOTEBOOK_START = time.time()

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    PLATFORM = "kaggle"
    REPO     = "/tmp/MoDiCF"
    RUNS_DST = "/kaggle/working/MoDiCF-runs"
elif "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    PLATFORM = "colab"
    from google.colab import drive
    drive.mount("/content/drive")
    REPO     = "/content/MoDiCF"
    RUNS_DST = "/content/drive/MyDrive/MoDiCF-runs"
    DATA_SRC = "/content/drive/MyDrive/MoDiCF-data"
else:
    raise RuntimeError("Neither Colab nor Kaggle detected - on a cluster use sbatch jobs/raid-cf/<ds>.slurm")

def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

print("platform:", PLATFORM, "| repo:", REPO, "| backup:", RUNS_DST)

## 1. Code + dependencies

If this cell downgrades numpy/torch in an interactive session, restart the runtime once and rerun from cell 0. (A Kaggle *Save & Run All* starts a fresh process, so it needs no restart.)

In [ ]:
if not os.path.isdir(REPO):
    sh(f"git clone -b {BRANCH} {REPO_URL} {REPO}")
else:
    sh(f"git -C {REPO} pull --ff-only")
os.chdir(REPO)

if MATCH_SERVER_TORCH:
    sh("pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118")
sh("pip install -q -r requirements.txt")
sh("which rsync >/dev/null || (apt-get -qq update && apt-get -qq install -y rsync)")
sh("nvidia-smi -L", check=False)

## 2. Data

In [ ]:
os.chdir(REPO)
os.makedirs("data", exist_ok=True)
if PLATFORM == "kaggle":
    # any attached input that holds <dataset>/train_mat
    hits = glob.glob(f"/kaggle/input/**/{DATASET}/train_mat", recursive=True)
    if not hits:
        raise FileNotFoundError(f"attach a Dataset containing {DATASET}/train_mat")
    DATA_SRC = os.path.dirname(os.path.dirname(hits[0]))
# copied, not linked: the retrieval neighbor cache is written into data/<dataset>/
sh(f"rsync -rL {DATA_SRC}/{DATASET} data/")
sh("ls data")

## 3. Restore logs / checkpoints

In [ ]:
os.chdir(REPO)
os.makedirs("logs", exist_ok=True)
os.makedirs("checkpoint", exist_ok=True)
if PLATFORM == "colab":
    os.makedirs(RUNS_DST, exist_ok=True)
    sources = [RUNS_DST]
else:
    # pretrain Dataset(s) and previous outputs of this notebook (.../MoDiCF-runs), oldest first
    sources = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/checkpoint", recursive=True)},
                     key=os.path.getmtime)
for src in sources:
    for sub in ("checkpoint", "logs"):
        if os.path.isdir(f"{src}/{sub}"):
            # -L: inputs may hold symlinks or resolved dirs; -u: never replace a newer file
            sh(f"rsync -rLu {src}/{sub}/ {sub}/")
print("restored from:", sources or "nothing")
sh(f"ls checkpoint/{DATASET} 2>/dev/null | head -50", check=False)

## 4. Background backup

`-L` turns the stage-1 symlink into a real dir in the backup (the scripts only check for `last_model.pth`); `*.tmp` skips a checkpoint that is mid-write.

In [ ]:
os.chdir(REPO)
os.makedirs(RUNS_DST, exist_ok=True)
BACKUP_CMD = f'rsync -rL --exclude="*.tmp" logs checkpoint {RUNS_DST}/'
sh('pkill -f "[M]oDiCF-backup-loop"', check=False)
subprocess.Popen(f"while true; do sleep {BACKUP_EVERY_S}; {BACKUP_CMD}; done # MoDiCF-backup-loop",
                 shell=True, start_new_session=True,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("backup loop ->", RUNS_DST)

## 5. Run the job

A new `SLURM_JOB_ID` per session gives each session its own log file: the script opens its log with `exec > "$LOG"` (truncate), so reusing an id would erase the epochs logged before the resume. Merge the pieces with cell 7 before aggregating.

In [ ]:
os.chdir(REPO)
JOB_ID = f"{PLATFORM}{time.strftime('%m%d%H%M')}"
env = dict(os.environ, SLURM_SUBMIT_DIR=REPO, SLURM_ARRAY_TASK_ID=str(SEED), SLURM_JOB_ID=JOB_ID)
cmd = f"bash jobs/raid-cf/{DATASET}.slurm {VARIANT} {EXTRA_ARGS}"
# own process group, so stopping it also stops the python child of the slurm script
job = subprocess.Popen(cmd, shell=True, env=env, start_new_session=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f"started {cmd!r} as job {JOB_ID} (pid {job.pid})")

if PLATFORM == "kaggle":
    import signal
    deadline = NOTEBOOK_START + KAGGLE_BUDGET_H * 3600
    while job.poll() is None and time.time() < deadline:
        time.sleep(60)
    if job.poll() is None:
        # checkpoints are written tmp+rename, so a stop mid-epoch is safe; the next run resumes
        print("time budget reached - stopping job")
        os.killpg(job.pid, signal.SIGTERM)
        time.sleep(30)
        if job.poll() is None:
            os.killpg(job.pid, signal.SIGKILL)
    print("job exit code:", job.poll())
    sh(BACKUP_CMD)

## 6. Monitor (Colab)

In [ ]:
os.chdir(REPO)
sh("tail -n 25 $(ls -t logs/raid-cf-*.out | head -1)", check=False)
print("running" if job.poll() is None else f"finished, exit code {job.returncode}")

## 7. Merge per-session logs, then aggregate

Concatenates every session's log of one run (same variant / dataset / hyperparameters / seed) in time order into `logs/merged/...-seed<N>-merged.out`, so `aggregate_seeds.py` sees one file per seed and still picks the best Recall@20 epoch.

In [ ]:
os.chdir(REPO)
from collections import defaultdict

runs = defaultdict(list)
for p in glob.glob("logs/raid-cf-*-seed*-*.out"):
    m = re.match(r"(raid-cf-.+-seed\d+)-[^-]+\.out$", os.path.basename(p))
    if m:
        runs[m.group(1)].append(p)

os.makedirs("logs/merged", exist_ok=True)
for key, paths in sorted(runs.items()):
    paths.sort(key=os.path.getmtime)
    with open(f"logs/merged/{key}-merged.out", "w") as out:
        for p in paths:
            out.write(open(p).read())
    print(f"{key}: {len(paths)} session log(s)")

sh(f'python scripts/aggregate_seeds.py "logs/merged/raid-cf-{VARIANT}-{DATASET}-*-seed*-merged.out"', check=False)

## 8. Final backup

In [ ]:
os.chdir(REPO)
sh(BACKUP_CMD)